# **ENSO Growth/Decay Rate Experiment Split by Eruption Seasonality - Models**

Runs Both: 

(1) Unconditional (all eruptions)

(2) Phase-conditioned (eruption-year phase = El Niño / Neutral / La Niña)

Output: 

-  CSV saved to ~/ENSO-GDR-Models-Seasonal.csv

In [14]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import cftime
from xarray import CFTimeIndex

# File Paths
model_paths = {
    "BCC-CSM1-1":     {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_bcc-csm1-1_r1i1p1_0850_2000_r240x120.nc", "year_offset": 0},
    "CCSM4":          {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_CCSM4_r1i1p1_0850_1850_r240x120.nc", "year_offset": 0},
    "CSIRO-Mk3L-1-2": {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_CSIRO-Mk3L-1-2_r1i1p1_0851_1850_r240x120.nc", "year_offset": 0},
    "GISS-E2-R":      {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_GISS-E2-R_r1i1p121_0850_1850_r240x120.nc", "year_offset": 0},
    "IPSL-CM5A-LR":   {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_IPSL-CM5A-LR_r1i1p1_0850_1850_r240x120.nc", "year_offset": 0},
    "MIROC-ES2L":     {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MIROC-ES2L_r1i1p1_0850_1849_r240x120.nc", "year_offset": 0},
    "MIROC-ESM":      {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MIROC-ESM_r1i1p1_0850_1849_r240x120.nc", "year_offset": 0},
    "MPI-ESM-P":      {"path": "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MPI-ESM-P_r1i1p1_0850_1849_r240x120.nc", "year_offset": 0},
    "MPI-ESM1-2":     {"path": "/g/data/ob22/ft3359/PMIPDATA/ts_Amon_MPI-ESM1-2-LR_past2k_r1i1p1f1_gn_700101-885012_regridded.nc", "year_offset": 6151},
    "MRI-ESM2-0":     {"path": "/g/data/ob22/ft3359/PMIPDATA/ts_Amon_MRI-ESM2-0_past1000_r1i1p1f1_gn_085001-184912_regridded.nc", "year_offset": 0},
}

# Eruption List with known seasonality
ERUPTIONS_RAW = [
    (939.0,  4.0,  1.0,  63.6, -1.0, 16.23, 4.97),
    (946.0, 11.0,  1.0,  42.0, -1.0,  1.72, 0.61),
    (1257.0, 7.0,  1.0,  -8.4,  1.4, 59.42, 10.86),
    (1477.0, 2.0,  1.0,  64.6, -1.0,  5.12, 1.61),
    (1510.0, 7.0, 25.0,  64.0, -1.0,  2.30, 0.83),
    (1585.0, 1.0, 10.0,  19.5, 10.6,  8.51, 2.34),
    (1595.0, 3.0,  1.0,   4.9,  0.8,  8.87, 1.51),
    (1600.0, 2.0, 17.0, -16.6,  2.0, 18.95, 4.03),
    (1640.0, 12.0, 26.0,  6.1,  2.8, 18.68, 4.28),
    (1667.0, 9.0, 23.0,  42.7, -1.0,  3.48, 1.11),
    (1673.0, 5.0, 20.0,   1.4,  0.7,  4.67, 0.82),
    (1707.0, 12.0, 16.0,  35.4, -1.0,  1.08, 0.40),
    (1721.0, 5.0, 11.0,  63.6, -1.0,  0.81, 0.36),
    (1739.0, 8.0, 19.0,  42.7, -1.0,  3.44, 1.09),
    (1755.0, 10.0, 17.0,  63.6, -1.0,  1.18, 0.43),
    (1766.0, 4.0,  5.0,  64.0, -1.0,  2.52, 0.75),
    (1783.0, 6.0, 15.0,  64.4, -1.0, 20.81, 7.04),
    (1815.0, 4.0, 10.0,  -8.0,  0.8, 28.08, 4.49),
    (1822.0, 10.0,  8.0,  -7.3, 10.0,  2.02, 0.79),
    (1835.0, 1.0, 20.0,  13.0,  2.0,  9.48, 2.21),
]
eruptions_df = pd.DataFrame(
    ERUPTIONS_RAW,
    columns=["yearCE", "month", "day", "lat", "hemi", "ssi", "sigma_ssi"],
)

eruptions_df["enso_year"] = np.where(
    eruptions_df["month"] >=7, eruptions_df["yearCE"], eruptions_df["yearCE"] -1,
).astype(int)

ERUPTION_YEARS = eruptions_df["enso_year"].to_numpy(dtype=int)

# Season classification
def classify_season(month: float) -> str:
    m = int(month)
    if m in (12, 1, 2):
        return "DJF"
    if m in (3, 4, 5):
        return "MAM"
    if m in (6, 7, 8):
        return "JJA"
    if m in (9, 10, 11):
        return "SON"
    return None

eruptions_df["season"] = eruptions_df["month"].apply(classify_season)

SEASONS = ["DJF", "MAM", "JJA", "SON"]
PHASES = ["El Niño", "Neutral", "La Niña"]
SEASON_BY_YEAR = dict(zip(eruptions_df["enso_year"], eruptions_df["season"]))

print(eruptions_df["season"].value_counts())

season
MAM    6
DJF    6
SON    4
JJA    4
Name: count, dtype: int64


In [15]:
# Settings

LAT_MIN, LAT_MAX = -5.0, 5.0
LON_MIN, LON_MAX = 190.0, 240.0

TROP_LAT_MIN, TROP_LAT_MAX = -20.0, 20.0

THRESHOLD = 0.5

LAG_MIN, LAG_MAX = -5, 5
FIT_LAG_MIN, FIT_LAG_MAX = 0, 3

EXCLUDE_MARGIN_YEARS = 5
N_MC = 4000
RNG_SEED = 42
MIN_EVENTS = 2

EPS = 1e-6
USE_ROBUST_CENTER = True

# OUTPUT SAVING
SAVE_CSV_SEASON = True
OUT_DIR_SEASON  = "/home/563/ft3359/FT-Honours/Honours_Paper/GDR/Seasonal_Analysis"
OUT_NAME_SEASON = "ENSO-GDR-Models-Seasonal.csv"

In [16]:
# Pipeline Functions

def load_sst_data(file_path, year_offset=0):
    ds = xr.open_dataset(file_path, use_cftime=True)
    sst = ds["ts"]

    if float(sst.max()) > 200:
        sst = sst - 273.15

    if year_offset:
        shifted = [t.replace(year=t.year - year_offset) for t in sst["time"].values]
        sst = sst.assign_coords(time=CFTimeIndex(shifted))

    t0 = sst["time"].values[0]
    start_str = "0850-01-01"
    end_str   = "1849-12-31"
    if isinstance(t0, cftime.Datetime360Day):
        end_str = "1849-12-30"

    sst = sst.sel(time=slice(start_str, end_str))
    return sst.chunk({"time": 365})


def area_weighted_mean_latlon(da: xr.DataArray) -> xr.DataArray:

    w = np.cos(np.deg2rad(da["lat"]))
    w = xr.DataArray(w, coords={"lat": da["lat"]}, dims=("lat",))
    return da.weighted(w).mean(dim=["lat", "lon"])


def calculate_relative_nino34_monthly(sst_1000):

    n34 = sst_1000.sel(lat=slice(LAT_MIN, LAT_MAX), lon=slice(LON_MIN, LON_MAX)).mean(dim=["lat", "lon"])

    trop = area_weighted_mean_latlon(sst_1000.sel(lat=slice(TROP_LAT_MIN, TROP_LAT_MAX)))

    n34_anom  = n34.groupby("time.month")  - n34.groupby("time.month").mean("time")
    trop_anom = trop.groupby("time.month") - trop.groupby("time.month").mean("time")

    rel = n34_anom - trop_anom

    return rel.rolling(time=3, center=True).mean()


def calculate_juljun_annual(monthly_3mo):
    t = monthly_3mo["time"]
    enso_year = xr.where(t.dt.month >= 7, t.dt.year, t.dt.year - 1)

    annual = monthly_3mo.groupby(enso_year).mean("time")

    years = annual["group"].values.astype(int)
    annual = annual.rename({"group": "time"}).assign_coords(time=years)
    return annual


def classify_phase_value(v, thr=0.5):
    if not np.isfinite(v):
        return None
    if v >= thr:
        return "El Niño"
    if v <= -thr:
        return "La Niña"
    return "Neutral"


# Rate Experiment Helpers

def in_eruption_margin_fast(years: np.ndarray, eruption_years: np.ndarray, margin: int) -> np.ndarray:
    years = years.astype(int)
    e_sorted = np.sort(eruption_years.astype(int))
    lo = np.searchsorted(e_sorted, years - margin, side="left")
    hi = np.searchsorted(e_sorted, years + margin, side="right")
    return (hi > lo)


def extract_event_windows_from_annual(years: np.ndarray, values: np.ndarray,
                                      event_years: np.ndarray, lag_min: int, lag_max: int) -> np.ndarray:
    y_to_i = {int(y): i for i, y in enumerate(years)}
    lags = np.arange(lag_min, lag_max + 1, dtype=int)
    mat = np.full((len(event_years), len(lags)), np.nan, dtype=float)
    for ei, ey in enumerate(event_years):
        ey = int(ey)
        for li, lag in enumerate(lags):
            yy = ey + int(lag)
            if yy in y_to_i:
                mat[ei, li] = values[y_to_i[yy]]
    return mat


def composite_mean(mat: np.ndarray) -> np.ndarray:
    return np.nanmean(mat, axis=0)


def fit_exponential_rate(t: np.ndarray, x: np.ndarray) -> float:
    t = np.asarray(t, float)
    x = np.asarray(x, float)
    ok = np.isfinite(t) & np.isfinite(x)
    t, x = t[ok], x[ok]
    if t.size < 3:
        return np.nan

    C = float(np.median(x[-2:])) if x.size >= 2 else float(x[-1])
    y = np.abs(x - C)
    good = np.isfinite(y) & (y > EPS)
    t2, y2 = t[good], y[good]
    if t2.size < 3:
        return np.nan

    logy = np.log(y2)
    A = np.vstack([t2, np.ones_like(t2)]).T
    slope, _ = np.linalg.lstsq(A, logy, rcond=None)[0]
    return float(slope)


def two_sided_p(null: np.ndarray, obs: float) -> float:
    null = np.asarray(null, float)
    null = null[np.isfinite(null)]
    if null.size == 0 or (not np.isfinite(obs)):
        return np.nan

    if USE_ROBUST_CENTER:
        c = float(np.median(null))
        dev = abs(obs - c)
        return float(np.mean(np.abs(null - c) >= dev))
    else:
        return float(np.mean(np.abs(null) >= abs(obs)))


def pick_control_years_phase_matched(years: np.ndarray,
                                     phase_by_year: dict,
                                     target_phase: str,
                                     eruption_years: np.ndarray,
                                     exclude_margin: int,
                                     n: int,
                                     rng: np.random.Generator) -> np.ndarray:
    years = np.asarray(years, int)
    in_margin = in_eruption_margin_fast(years, eruption_years, exclude_margin)
    ok_years = years[(~in_margin) & (~np.isin(years, eruption_years))]

    pool = np.array([y for y in ok_years if phase_by_year.get(int(y), None) == target_phase], dtype=int)
    if pool.size == 0:
        return np.array([], dtype=int)
    replace = pool.size < n
    return rng.choice(pool, size=n, replace=replace).astype(int)

In [17]:
# Main — per-model, non-pooled Season x Phase experiment

lags = np.arange(LAG_MIN, LAG_MAX + 1, dtype=int)
fit_mask = (lags >= FIT_LAG_MIN) & (lags <= FIT_LAG_MAX)
t_fit = lags[fit_mask].astype(float)

rows_season = []

for model_idx, (model_name, cfg) in enumerate(model_paths.items()):
    print(f"\nLoading {model_name}...")
    try:
        sst = load_sst_data(cfg["path"], year_offset=cfg.get("year_offset", 0))
        rel_monthly = calculate_relative_nino34_monthly(sst)
        rel_annual_c = calculate_juljun_annual(rel_monthly)
    except Exception as e:
        print(f"  Failed {model_name}: {type(e).__name__}: {e}")
        continue

    years_avail = rel_annual_c.time.values.astype(int)
    vals_annual = rel_annual_c.values.astype(float)
    years_set = set(int(y) for y in years_avail)

    erupt_years = np.array([y for y in ERUPTION_YEARS if int(y) in years_set], dtype=int)

    phase_by_year = {}
    for y, v in zip(years_avail, vals_annual):
        ph = classify_phase_value(float(v), THRESHOLD)
        if ph is not None:
            phase_by_year[int(y)] = ph

    rng = np.random.default_rng(RNG_SEED + model_idx)

    for season in SEASONS:
        for phase in PHASES:
            evsp = np.array(
                [y for y in erupt_years
                 if SEASON_BY_YEAR.get(int(y)) == season
                 and phase_by_year.get(int(y)) == phase],
                dtype=int,
            )
            case_label = f"{season} | {phase}"

            if evsp.size < MIN_EVENTS:
                rows_season.append(dict(dataset=model_name, season=season, phase=phase, case=case_label, N_events=int(evsp.size), r_yr1=np.nan, tau_yr=np.nan, p_value=np.nan))
                continue

            matE = extract_event_windows_from_annual(years_avail, vals_annual, evsp, LAG_MIN, LAG_MAX)
            compE = composite_mean(matE)
            r_obs = fit_exponential_rate(t_fit, compE[fit_mask])
            tau = (-1.0 / r_obs) if (np.isfinite(r_obs) and r_obs < 0) else np.nan

            r_null = np.full(N_MC, np.nan, dtype=float)
            for i in range(N_MC):
                ctrl = pick_control_years_phase_matched(
                    years_avail, phase_by_year, phase, erupt_years, EXCLUDE_MARGIN_YEARS, evsp.size, rng
                )
                if ctrl.size == 0:
                    continue
                matC = extract_event_windows_from_annual(years_avail, vals_annual, ctrl, LAG_MIN, LAG_MAX)
                compC = composite_mean(matC)
                r_null[i] = fit_exponential_rate(t_fit, compC[fit_mask])

            p_mc = two_sided_p(r_null, r_obs)

            rows_season.append(dict(dataset=model_name, season=season, phase=phase, case=case_label, N_events=int(evsp.size), r_yr1=float(r_obs) if np.isfinite(r_obs) else np.nan, tau_yr=float(tau) if np.isfinite(tau) else np.nan,
                                     p_value=float(p_mc) if np.isfinite(p_mc) else np.nan))


out_season = pd.DataFrame(rows_season)

print(out_season[["dataset","season","phase","N_events","r_yr1","tau_yr","p_value"]].to_string(index=False))

# Save

if SAVE_CSV_SEASON:
    os.makedirs(OUT_DIR_SEASON, exist_ok=True)
    out_path_season = os.path.join(OUT_DIR_SEASON, OUT_NAME_SEASON)
    out_season[["dataset", "season", "phase", "case", "N_events", "r_yr1", "tau_yr", "p_value"]].to_csv(out_path_season, index=False)
    print("\nSaved CSV:", out_path_season)


Loading BCC-CSM1-1...


/jobfs/178626876.gadi-pbs/ipykernel_3081568/3485153491.py:4: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)



Loading CCSM4...


/jobfs/178626876.gadi-pbs/ipykernel_3081568/3485153491.py:4: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)



Loading CSIRO-Mk3L-1-2...


...


Loading MPI-ESM1-2...


/jobfs/178626876.gadi-pbs/ipykernel_3081568/3485153491.py:4: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)



Loading MRI-ESM2-0...


/jobfs/178626876.gadi-pbs/ipykernel_3081568/3485153491.py:4: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)
/jobfs/178626876.gadi-pbs/ipykernel_3081568/3485153491.py:91: RuntimeWarning: Mean of empty slice
  return np.nanmean(mat, axis=0)


       dataset season   phase  N_events     r_yr1    tau_yr  p_value
    BCC-CSM1-1    DJF El Niño         2 -0.084700 11.806362 0.273000
    BCC-CSM1-1    DJF Neutral         3 -0.579085  1.726861 0.229500
    BCC-CSM1-1    DJF La Niña         1       NaN       NaN      NaN
    BCC-CSM1-1    MAM El Niño         1       NaN       NaN      NaN
    BCC-CSM1-1    MAM Neutral         3  0.455060       NaN 0.376000
    BCC-CSM1-1    MAM La Niña         2 -0.172601  5.793720 0.436750
    BCC-CSM1-1    JJA El Niño         0       NaN       NaN      NaN
    BCC-CSM1-1    JJA Neutral         4  0.057794       NaN 0.939000
    BCC-CSM1-1    JJA La Niña         0       NaN       NaN      NaN
    BCC-CSM1-1    SON El Niño         1       NaN       NaN      NaN
    BCC-CSM1-1    SON Neutral         3 -0.054215 18.445130 0.847750
    BCC-CSM1-1    SON La Niña         0       NaN       NaN      NaN
         CCSM4    DJF El Niño         1       NaN       NaN      NaN
         CCSM4    DJF Neutral     